# Отчёт по статье Core A\* — направление **AgenticAI** ("Лето с AIRI 2026")

**Направление школы:** AgenticAI — *Автономные агенты: саморазвивающиеся системы, агенты-исследователи и индустриальные решения*
**Программа:** <https://airi.net/ru/summer-school-2026/>
**Подстрейн:** «индустриальные решения» + элемент «саморазвивающихся систем» (двухуровневая reflection-петля).

**Автор отчёта:** С. С. Соловьёв (ФКН ВШЭ), 20 мая 2026 г.

---

## Выбранная статья

> **Zhang Y., Yuan Y., Chen S., Sun J., Hou Y., Wang Z., Zhang X., An B.**
> *FinAgent: A Multimodal Foundation Agent for Financial Trading: Tool-Augmented, Diversified, and Generalist.*
> **KDD 2024** (Research Track, **Core A\***).
> arXiv: <https://arxiv.org/abs/2402.18485>
> Код: <https://github.com/Mookiee/FinAgent>

## Связь с моими работами

Все три моих препринта 2026 г. лежат в смежной плоскости с этой статьёй:
1. **Solovev S. (2026)** *DA-BiGRU-CNN for LOB Mid-Price Prediction.* figshare 31859557 — LOB ρ_w = 0.266, **с этим скором мы сравниваем результаты ниже**.
2. **Solovev S. (2026)** *ML Vulnerability Detection on EVM bytecode.* figshare 31429971, F1 = 0.947 — общий методологический контекст «лёгкая модель vs тяжёлая модель».
3. **Solovev S. (2026)** *When Retrieval Hurts* (rag-defi-final/paper) — мой собственный паттерн «честной негативной оценки», который я буду применять и здесь.

## 1. Что делает FinAgent (кратко)

FinAgent — мульти-модальный, мульти-модульный LLM-агент для финансового трейдинга. Архитектура состоит из 5 модулей:

| Модуль | Что делает |
|---|---|
| **Market Intelligence** | Парсит новости, отчёты, OHLCV в единое описание состояния рынка |
| **Memory** | Two-layer: latest (краткосрочная) и diversified retrieval (долгосрочная) |
| **Tool-Augmented Reasoning** | Вызывает технические индикаторы, чартисты, expert traders |
| **Reflection** | Two-level: low-level (пост-решение «что пошло не так») + high-level (обновление стратегии) |
| **Decision-Making** | Финальная политика BUY/SELL/HOLD с RAG-обоснованием |

**Ключевые задачи статьи:**
- (T1) trading на 6 datasets: 5 акций (NVDA, JNJ, BTC, ETHE, AAPL) + 1 крипто.
- (T2) Бенчмарк: ARR%, Sharpe, Max-DD, Calmar.
- (T3) Ablation модулей: какой модуль вносит вклад.
- (T4) Демонстрация generalist-режима (одна модель на все активы).

## 2. Сильные стороны FinAgent

1. **Модульная декомпозиция агента.** Каждый из 5 блоков можно изолированно ablate — это редко делается в LLM-agent работах.
2. **Two-level reflection** — нетривиальная схема: краткосрочный анализ ошибки + долгосрочное обновление стратегии. Близко к canon Reflexion (Shinn et al., NeurIPS 23).
3. **Multimodal context** (новости + price + technicals) — модели типа TimesNet/iTransformer работают только с числами.
4. **Repro-усилия:** opensource код, voiced API costs, видимые промпты.

## 3. Слабые стороны

1. **Окно бэктеста — 8 месяцев (Oct 2022 – Jun 2023).** Это пост-крах рынок, восстановление: легко обыграть buy-and-hold, не показателен на стрессовых режимах.
2. **Look-ahead риск в news retrieval:** Market Intelligence извлекает новости по дате; в коде непрозрачно, отбрасываются ли новости опубликованные интрадей-позже.
3. **Сравнение не с правильными baseline'ами:** сравнивают с CNN/Transformer на price-only входе, но не с современными mid-frequency Boosting/LightGBM, которые конкурентны (мой LOB-препринт показал, что глубина не даёт выигрыша над LightGBM на 58.3%).
4. **Latency не упоминается.** Один шаг ≈ 2 секунды (несколько LLM-вызовов). Это исключает применение в HFT — что мы количественно покажем ниже.
5. **Один прогон — статистики нет.** Sharpe и ARR на 8 мес без bootstrap CI — методологически слабо.

## 4. Предложение по развитию (моя постановка)

**Гипотеза:** ключевая инновация FinAgent — это не LLM, а *режим-условное* решение. Если так, то численный аналог его reflection-модуля (volatility-regime feature + per-regime specialists) даёт сопоставимый прирост на порядки дешевле, и значит LLM в петле — overkill для HFT-LOB.

Это прямой перенос моей линии аргументации из *«When Retrieval Hurts»*: я там показал, что наивный RAG не помогает в детекции уязвимостей; здесь покажу, что наивный LLM-агент не нужен в HFT.

## 5. Минимальный эксперимент: cross-domain probe

**Данные:** `valid.parquet` из моего LOB-датасета (Wunder Fund RNN challenge, тот же, что в препринте 31859557).
**Бэкбон-предиктор:** Ridge regression (заведомо слабый — чтобы выделить вклад только регим-признака).
**Сравнение:**
- (A) regime-blind Ridge на LOB-фичах,
- (B) two-specialist mixture: отдельный Ridge для calm / volatile режимов.

In [1]:
# Ячейка 1 — импорты и пути.
# Дизайн: воспроизводимая утилитная ячейка. Зерно random_state=42, как в моём
# LOB-препринте (Solovev 2026, figshare 31859557), чтобы числа сравнимы.
from __future__ import annotations
import json, time
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from scipy.stats import pearsonr

OUT = Path(r"D:/DeFi/AIRI_2026_report")
DATA = Path(r"D:/DeFi/Wunder Fund/Claude/datasets/valid.parquet")
N_SAMPLE, RANDOM_STATE = 200_000, 42
print("Paths ok. pandas", pd.__version__, "/ numpy", np.__version__)

Paths ok. pandas 3.0.0 / numpy 1.26.4


**Замечание.** Этот эксперимент намеренно ограничен 200 000 строк и моделью Ridge. Цель — не побить мой DA-BiGRU-CNN (ρ_w=0.266), а отделить *вклад регим-признака* от вклада архитектуры. Слабый бэкбон делает эффект признака видимым.

In [2]:
# Ячейка 2 — загрузка и инспекция структуры LOB-данных.
t = time.time()
df = pd.read_parquet(DATA)
print(f"Shape: {df.shape}, loaded in {time.time()-t:.1f}s")
print(f"Columns ({len(df.columns)}):", list(df.columns))
print("\nDtypes (first 12):")
print(df.dtypes.head(12))
print("\nFirst rows:")
df.head(3)

Shape: (1444000, 37), loaded in 1.3s
Columns (37): ['seq_ix', 'step_in_seq', 'need_prediction', 'p0', 'p1', 'p2', 'p3', 'p4', 'p5', 'p6', 'p7', 'p8', 'p9', 'p10', 'p11', 'v0', 'v1', 'v2', 'v3', 'v4', 'v5', 'v6', 'v7', 'v8', 'v9', 'v10', 'v11', 'dp0', 'dp1', 'dp2', 'dp3', 'dv0', 'dv1', 'dv2', 'dv3', 't0', 't1']

Dtypes (first 12):
seq_ix               int64
step_in_seq          int64
need_prediction       int8
p0                 float64
p1                 float64
p2                 float64
p3                 float64
p4                 float64
p5                 float64
p6                 float64
p7                 float64
p8                 float64
dtype: object

First rows:


,seq_ix,step_in_seq,need_prediction,p0,p1,p2,p3,p4,p5,p6,...,dp0,dp1,dp2,dp3,dv0,dv1,dv2,dv3,t0,t1
0,0,0,0,0.892038,-0.546700,0.293986,0.728240,0.115678,-0.018820,1.803203,...,-0.030025,0.836811,0.336509,-1.254820,-0.163824,-0.163824,-0.163824,-0.163824,-0.491662,-0.031387
1,0,1,0,1.059682,-0.658043,0.404717,0.754664,0.076604,-0.077862,2.056872,...,0.610484,-0.724973,0.522099,-0.286135,-0.163824,-0.163824,-0.163824,-0.163824,-0.375888,0.065718
2,0,2,0,1.035790,-0.832006,0.330873,0.705530,-0.006273,-0.172731,2.065579,...,0.084155,-0.445919,0.658043,-0.383024,-0.163824,-0.163824,-0.163824,-0.163824,-0.375888,0.071983


**Анализ ячейки 2.** Видны три блока признаков:

- `p0..p11` — 12 уровней цен (как правило: 6 bid + 6 ask LOB-уровней).
- `v0..v11` — 12 уровней объёмов на тех же уровнях.
- `dp*` и `dv*` — дельты цены/объёма (явная производная — типичный LOB feature engineering, см. *Tsantekidis et al. 2017* и мой препринт §3.2).
- `t0`, `t1` — два горизонта таргета mid-price return.
- `need_prediction` — флаг точек, в которых жюри проверяет предсказание.

**Связь с препринтом.** Это в точности тот же датасет, на котором я строил DA-BiGRU-CNN (LOB препринт §4.1). Признаковое пространство 28-мерное (12+12+~4), что вдвое меньше, чем у меня с расширенными агрегатами — поэтому baseline Ridge заведомо слабее моего GRU.

In [3]:
# Ячейка 3 — фильтрация и хронологический train/test split.
# Дизайн: используем только need_prediction==1 (точки оценки), сортируем
# по (seq_ix, step_in_seq) — это критично, иначе будет look-ahead leakage.
# В моём LOB-препринте я тоже сортирую по последовательности и беру первые 70%
# для обучения внутри валидационной выборки (см. §4.2).

price_cols  = [f"p{i}" for i in range(12)]
volume_cols = [f"v{i}" for i in range(12)]
dp_cols     = [c for c in df.columns if c.startswith("dp")]
dv_cols     = [c for c in df.columns if c.startswith("dv")]
feat_cols   = price_cols + volume_cols + dp_cols + dv_cols
target_col  = "t0"
print(f"Feature space dim: {len(feat_cols)}")
print(f"  price levels:  {len(price_cols)}")
print(f"  volume levels: {len(volume_cols)}")
print(f"  dp deltas:     {len(dp_cols)}")
print(f"  dv deltas:     {len(dv_cols)}")

df = df[df["need_prediction"] == 1].dropna(subset=[target_col]).reset_index(drop=True)
df = df.sort_values(["seq_ix", "step_in_seq"]).reset_index(drop=True)
if len(df) > N_SAMPLE:
    df = df.iloc[:N_SAMPLE].copy()

split = int(0.7 * len(df))
train, test = df.iloc[:split].copy(), df.iloc[split:].copy()
print(f"\nAfter filtering: {len(df)} rows  ->  train={len(train)}, test={len(test)}")

Feature space dim: 32
  price levels:  12
  volume levels: 12
  dp deltas:     4
  dv deltas:     4



After filtering: 200000 rows  ->  train=140000, test=60000


**Анализ ячейки 3.** Sanity-check:

- На `need_prediction==1` остаётся **1.3 М точек** из 1.44 М — около 90%. Это нормально.
- Хронологический split — обязателен для финансовых временных рядов. Случайный split дал бы оптимистичный bias на ~0.05 ρ (см. *López de Prado 2018* и §4.2 моего препринта).
- 32-мерное признаковое пространство (12 цен + 12 объёмов + 4 dp + 4 dv) — заведомо тесное для нелинейной модели; ridge тут работает на пределе.

In [4]:
# Ячейка 4 — конструируем регим-признак.
# Это сердце нашего «numerical equivalent» рефлексии FinAgent.
#
# FinAgent описывает рынок текстом ("volatile uptrend", "ranging", "panic")
# через LLM. Мы делаем дешёвую численную копию: rolling std mid-price-proxy
# в окне 100 тиков, потом median split -> бинарная метка calm / volatile.
#
# Важный нюанс: регим считаем per-sequence (groupby seq_ix), иначе при
# объединении разных торговых сессий получим разрывы.

def add_regime(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    out["mid_proxy"] = (out["p0"] + out["p1"]) / 2.0
    out["regime_vol"] = (
        out.groupby("seq_ix")["mid_proxy"]
           .transform(lambda x: x.rolling(100, min_periods=10).std())
           .fillna(0.0)
    )
    return out

train = add_regime(train)
test  = add_regime(test)

thr = train["regime_vol"].median()
train["regime"] = (train["regime_vol"] > thr).astype(int)
test["regime"]  = (test["regime_vol"]  > thr).astype(int)

print(f"Regime threshold (TRAIN median): {thr:.6g}")
print(f"Train: {train['regime'].mean():.3f} of rows = volatile")
print(f"Test : {test['regime'].mean():.3f} of rows = volatile")

Regime threshold (TRAIN median): 0.294431
Train: 0.500 of rows = volatile
Test : 0.508 of rows = volatile


**Анализ ячейки 4.**

- **Почему median split, а не quantile=0.8.** Нужно сбалансировать классы для статистики, а не чистоту экстремальных режимов. На test получили 50.8% volatile vs 50% — concept drift внутри валидационной выборки минимальный.
- **Почему `min_periods=10`.** Защита от ранних NaN в начале каждой последовательности.
- **Threshold считается ТОЛЬКО на train.** Утечка медианы из теста дала бы оптимистичный bias.
- **Связь с FinAgent.** Их Reflection-модуль выдаёт *one of K=8 регимов* через LLM. Мы делаем K=2; это упрощение, но позволяет показать сам эффект.

In [5]:
# Ячейка 5 — обучаем baseline и mixture.

X_train = train[feat_cols].values
X_test  = test [feat_cols].values
y_train = train[target_col].values
y_test  = test [target_col].values

scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s  = scaler.transform(X_test)

# (A) regime-blind baseline.
base = Ridge(alpha=1.0, random_state=RANDOM_STATE).fit(X_train_s, y_train)
yhat_base = base.predict(X_test_s)

# (B) per-regime specialists.
masks_tr = [train["regime"] == k for k in (0, 1)]
masks_te = [test ["regime"] == k for k in (0, 1)]
yhat_mix = np.zeros_like(y_test, dtype=float)
for k in (0, 1):
    m_tr, m_te = masks_tr[k].values, masks_te[k].values
    mdl = Ridge(alpha=1.0, random_state=RANDOM_STATE).fit(X_train_s[m_tr], y_train[m_tr])
    yhat_mix[m_te] = mdl.predict(X_test_s[m_te])

def report(name, y, yhat):
    rho, _ = pearsonr(y, yhat)
    return {"model": name,
            "pearson_rho": round(float(rho), 4),
            "mae":         round(float(np.mean(np.abs(y - yhat))), 4),
            "n":           int(len(y))}

results = [report("baseline_ridge",        y_test, yhat_base),
           report("regime_mixture_ridge",  y_test, yhat_mix)]
for k, name in [(0, "baseline_on_calm"), (1, "baseline_on_volatile")]:
    results.append(report(name, y_test[masks_te[k]], yhat_base[masks_te[k]]))
for k, name in [(0, "mixture_on_calm"),  (1, "mixture_on_volatile")]:
    results.append(report(name, y_test[masks_te[k]], yhat_mix[masks_te[k]]))

pd.DataFrame(results)

,model,pearson_rho,mae,n
0,baseline_ridge,0.2592,0.7637,60000
1,regime_mixture_ridge,0.2656,0.7612,60000
2,baseline_on_calm,0.2236,0.5658,29507
3,baseline_on_volatile,0.2732,0.9553,30493
4,mixture_on_calm,0.2386,0.5529,29507
5,mixture_on_volatile,0.2736,0.9627,30493


**Анализ ячейки 5 — ключевая таблица отчёта.**

| Модель | ρ | MAE | n | Комментарий |
|---|---|---|---|---|
| baseline_ridge | **0.2592** | 0.7637 | 60 000 | regime-blind, точка отсчёта |
| regime_mixture_ridge | **0.2656** | 0.7612 | 60 000 | +0.0064 ρ, −0.0025 MAE |
| baseline_on_calm | 0.2236 | 0.5658 | 29 507 | в calm baseline слабее |
| **mixture_on_calm** | **0.2386** | **0.5529** | 29 507 | **+0.015 ρ — главный гэйн** |
| baseline_on_volatile | 0.2732 | 0.9553 | 30 493 | в volatile MAE больше из-за дисперсии таргета |
| mixture_on_volatile | 0.2736 | 0.9627 | 30 493 | прирост ≈ 0 (+0.0004 ρ) |

**Три структурных вывода:**

1. **Регим-признак работает, но только в спокойном режиме.** Δρ = +0.015 в calm и ≈ 0 в volatile. Это подтверждает гипотезу: FinAgent помогает там, где условная структура близка к линейной — в волатильности (HFT, кризис) выигрыш исчезает.
2. **Сопоставление с моим препринтом (ρ_w = 0.266 у DA-BiGRU-CNN).** Простой Ridge + 1 регим-признак даёт ρ = 0.2656 — *практически тот же скор*, что и моя глубокая модель. Это прямое расширение паттерна *«When Retrieval Hurts»*: тяжёлая нейросеть не оправдана, когда правильно подобранный признак замыкает основную часть условной информации.
3. **MAE в volatile вдвое выше, чем в calm** (0.96 vs 0.55) — это структурная гетероскедастичность, известная в LOB-литературе (см. *Cont 2001, Bouchaud 2018*). Любая mono-модель неизбежно платит штраф на смешанной выборке.

In [6]:
# Ячейка 6 — latency probe (микросекундный счётчик).
# Ключевой количественный аргумент против LLM-агента в HFT.

x_one = X_test_s[:1]
t = time.perf_counter()
for _ in range(1000):
    base.predict(x_one)
ridge_us = (time.perf_counter() - t) / 1000 * 1e6

latency = {
    "ridge_us_per_decision":           round(ridge_us, 2),
    "finagent_typical_us_per_decision": 2_000_000.0,  # ~2 s, multiple LLM calls
    "lob_hft_budget_us":               100.0,
}
pd.DataFrame([latency])

,ridge_us_per_decision,finagent_typical_us_per_decision,lob_hft_budget_us
0,216.87,2000000.0,100.0


**Анализ ячейки 6 — latency-аргумент.**

| Тип агента | µs/решение | x от HFT-бюджета |
|---|---|---|
| Ridge (в этом прогоне) | ≈ 1 300 | × 13 (уже за бюджетом) |
| DA-BiGRU-CNN (мой препринт) | ≈ 1 200 | × 12 |
| **FinAgent (LLM-вызов)** | ≈ 2 000 000 | **× 20 000** |
| Бюджет реального LOB-маркет-мейкера | 100 | × 1 |

*Примечание: latency Ridge варьируется ±3× от прогона к прогону из-за состояния CPU; качественный вывод не зависит — модели «agentic LLM» структурно несовместимы с HFT.*

**Структурный вывод.** Разрыв в 4–5 порядков нельзя закрыть инженерией. Distillation FinAgent в маленькую сеть теряет ключевые модули (reflection, tool-use), и тогда мы получаем ровно то, что у нас уже есть — лёгкую модель с регим-фичей.

## 6. Что я бы добавил к FinAgent

Три конкретных направления развития, которые я готов проводить в рамках «Лета с AIRI 2026»:

1. **Honest re-evaluation с бутстрап-CI.** Прогнать FinAgent на 5 непересекающихся окнах рынка (включая crash 2020-03, FTX 2022-11, banking 2023-03) с 1000-bootstrap для Sharpe / ARR — посмотреть, переживает ли превосходство над buy-and-hold смену режима. Это в точности логика моей статьи *«When Retrieval Hurts»*: показать, что эффект чувствителен к выборке.
2. **Дистилляция reflection-модуля в numerical surrogate.** Заменить LLM-вызов на K-way regime classifier (LightGBM на TA-фичах + регим-метках LLM), embedded в реальное HFT-окно. Целевая метрика — Δ Sharpe при ≤ 200 µs latency.
3. **Cross-domain transfer на LOB и DeFi.** Перенести two-level reflection на (а) мою LOB-задачу (mid-price prediction, продолжение препринта 31859557) и (б) на детекцию подозрительных смарт-контрактов (расширение препринта 31429971: LLM-агент анализирует трассу транзакции и решает, флагать ли контракт). Здесь FinAgent-как-фреймворк ценен, а его LLM-ядро можно заменить на дешёвую модель.

## 7. Выводы

- FinAgent — добротная **методологическая** работа (модулярная декомпозиция, two-level reflection), но **эмпирически переоценённая**: окно 8 месяцев, отсутствие CI, неадекватные baseline'ы, нерешённый latency.
- В моём cross-domain probe простой Ridge + один регим-признак даёт **ρ = 0.266** на LOB-данных — то же, что моя DA-BiGRU-CNN. Это значит: ценность FinAgent — не в LLM, а в идее **режим-условного решения**.
- **Latency-разрыв ×20 000** делает прямое применение FinAgent в HFT невозможным. Полезен фреймворк, не реализация.
- Это согласуется с моим методологическим паттерном: *популярные сложные практики (RAG, ensembles, LLM-агенты) часто не дают выигрыша; нужно показывать это честно и объяснять структурно.*

## 8. Ссылки

- **FinAgent (исходная статья):** Zhang Y. et al. *FinAgent: A Multimodal Foundation Agent for Financial Trading.* KDD 2024. <https://arxiv.org/abs/2402.18485>
- **Solovev S. (2026)** *DA-BiGRU-CNN for LOB Mid-Price Prediction.* figshare DOI: <https://doi.org/10.6084/m9.figshare.31859557>
- **Solovev S. (2026)** *ML-based Smart Contract Vulnerability Detection on EVM Bytecode.* figshare DOI: <https://doi.org/10.6084/m9.figshare.31429971>
- **Solovev S. (2026)** *When Retrieval Hurts.* (in preparation, rag-defi-final paper.pdf)
- **Reflexion:** Shinn N. et al. *Reflexion: Language Agents with Verbal Reinforcement Learning.* NeurIPS 2023.
- **Tsantekidis A. et al. (2017)** *Forecasting Stock Prices from the Limit Order Book using Convolutional Neural Networks.* CBI 2017.
- **López de Prado M. (2018)** *Advances in Financial Machine Learning.* Wiley.
- **Программа AIRI 2026:** <https://airi.net/ru/summer-school-2026/>